In [1]:
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load saved models
with open('nhanes_data/diabetes_model.json') as f:
    dm = json.load(f)
with open('nhanes_data/cvd_model.json') as f:
    cm = json.load(f)

# ─────────────────────────────────────────────────────────
# Helper: score a meal using the ML model
# Features must match FEATURE_COLS order
# ─────────────────────────────────────────────────────────
def ml_score(model_data, features_dict):
    feature_cols = model_data['feature_cols']
    x = np.array([features_dict.get(c, 0) for c in feature_cols])
    # Standardise using saved scaler parameters
    x_scaled = (x - np.array(model_data['scaler_mean'])) / np.array(model_data['scaler_scale'])
    # Logistic regression prediction
    logit = np.dot(x_scaled, model_data['coefficients']) + model_data['intercept']
    prob = 1 / (1 + np.exp(-logit))
    return round(prob * 100)  # 0-100 score

# ─────────────────────────────────────────────────────────
# CRITERION 1: Directionality check
# High-risk meal should score >= 60
# Low-risk meal should score <= 30
# ─────────────────────────────────────────────────────────
print('=' * 60)
print('CRITERION 1: Directionality')
print('=' * 60)

# High-risk reference: white rice + ghee meal
# GL ~86 (but our feature is per-day, so scale down to per-meal fraction)
# We test the model on a 'high-risk day' nutritional profile
high_risk_day = {
    'feature_glycemic_load':      280,   # high GL day
    'feature_refined_carb_share':  0.85,  # mostly refined carbs
    'feature_fiber_per_1000kcal':  4.0,   # low fiber
    'feature_protein_pct_energy':  0.10,  # low protein
    'feature_sfa_pct_energy':      0.14,  # high SFA
    'feature_mufa_sfa_ratio':      0.4,   # poor fat quality (ghee)
    'feature_sodium_mg':           3500,  # moderate-high sodium
}

# Low-risk reference: dal + whole wheat roti + vegetables
low_risk_day = {
    'feature_glycemic_load':      90,
    'feature_refined_carb_share':  0.10,
    'feature_fiber_per_1000kcal':  18.0,
    'feature_protein_pct_energy':  0.18,
    'feature_sfa_pct_energy':      0.04,
    'feature_mufa_sfa_ratio':      2.8,
    'feature_sodium_mg':           1200,
}

high_d = ml_score(dm, high_risk_day)
high_c = ml_score(cm, high_risk_day)
low_d  = ml_score(dm, low_risk_day)
low_c  = ml_score(cm, low_risk_day)

print(f'High-risk day — Diabetes: {high_d}/100  CVD: {high_c}/100')
print(f'Low-risk day  — Diabetes: {low_d}/100  CVD: {low_c}/100')

c1_diabetes = high_d >= 60 and low_d <= 30
c1_cvd      = high_c >= 60 and low_c <= 30
print(f'Diabetes directionality: {"PASS" if c1_diabetes else "FAIL"}')
print(f'CVD directionality:      {"PASS" if c1_cvd else "FAIL"}')

# ─────────────────────────────────────────────────────────
# CRITERION 2: Coefficient signs (already checked in Steps 46–47)
# ─────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('CRITERION 2: Coefficient signs')
print('=' * 60)
expected_signs_d = ['+', '+', '-', '-', '+', '-', '?']
expected_signs_c = ['+', '+', '-', '-', '+', '-', '+']
feature_names = dm['feature_cols']

d_signs_ok = all(
    (exp == '?') or
    (exp == '+' and coef > 0) or
    (exp == '-' and coef < 0)
    for exp, coef in zip(expected_signs_d, dm['coefficients'])
)
c_signs_ok = all(
    (exp == '+' and coef > 0) or (exp == '-' and coef < 0)
    for exp, coef in zip(expected_signs_c, cm['coefficients'])
)
print(f'Diabetes coefficient signs: {"PASS" if d_signs_ok else "FAIL"}')
print(f'CVD coefficient signs:      {"PASS" if c_signs_ok else "FAIL"}')

# ─────────────────────────────────────────────────────────
# CRITERION 3: Asian subsample AUC >= 0.70
# ─────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('CRITERION 3: Asian subsample AUC')
print('=' * 60)
d_asian_auc = dm.get('asian_auc')
c_asian_auc = cm.get('asian_auc')
print(f'Diabetes Asian AUC: {d_asian_auc}')
print(f'CVD Asian AUC:      {c_asian_auc}')
c3_d = d_asian_auc and d_asian_auc >= 0.70
c3_c = c_asian_auc and c_asian_auc >= 0.70
print(f'Diabetes: {"PASS" if c3_d else "FAIL / INSUFFICIENT DATA"}')
print(f'CVD:      {"PASS" if c3_c else "FAIL / INSUFFICIENT DATA"}')

# ─────────────────────────────────────────────────────────
# CRITERION 4: Spearman correlation with rule-based scores
# on 20 reference South Asian meals
# ─────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('CRITERION 4: Correlation with rule-based scores')
print('=' * 60)

# 20 reference meals with their rule-based scores (from your app)
# You need to run these meals through your web app and record the scores
# Format: (meal_name, rule_diabetes, rule_cvd, feature_dict)
# Replace the scores below with actual values from your app after Step 50
reference_meals = [
    ('Dal tadka',         22, 18, {'feature_glycemic_load':90,  'feature_refined_carb_share':0.05, 'feature_fiber_per_1000kcal':18, 'feature_protein_pct_energy':0.17, 'feature_sfa_pct_energy':0.05, 'feature_mufa_sfa_ratio':2.1, 'feature_sodium_mg':800}),
    ('Rajma chawal',      38, 22, {'feature_glycemic_load':160, 'feature_refined_carb_share':0.55, 'feature_fiber_per_1000kcal':12, 'feature_protein_pct_energy':0.14, 'feature_sfa_pct_energy':0.04, 'feature_mufa_sfa_ratio':1.9, 'feature_sodium_mg':600}),
    ('Chole bhature',     65, 35, {'feature_glycemic_load':230, 'feature_refined_carb_share':0.75, 'feature_fiber_per_1000kcal':8,  'feature_protein_pct_energy':0.12, 'feature_sfa_pct_energy':0.07, 'feature_mufa_sfa_ratio':1.2, 'feature_sodium_mg':900}),
    ('Palak paneer+roti', 30, 42, {'feature_glycemic_load':110, 'feature_refined_carb_share':0.20, 'feature_fiber_per_1000kcal':14, 'feature_protein_pct_energy':0.18, 'feature_sfa_pct_energy':0.12, 'feature_mufa_sfa_ratio':0.5, 'feature_sodium_mg':700}),
    ('White rice+ghee',   72, 68, {'feature_glycemic_load':280, 'feature_refined_carb_share':0.90, 'feature_fiber_per_1000kcal':3,  'feature_protein_pct_energy':0.08, 'feature_sfa_pct_energy':0.14, 'feature_mufa_sfa_ratio':0.4, 'feature_sodium_mg':400}),
    ('Idli sambar 2srv',  28, 15, {'feature_glycemic_load':130, 'feature_refined_carb_share':0.50, 'feature_fiber_per_1000kcal':11, 'feature_protein_pct_energy':0.13, 'feature_sfa_pct_energy':0.03, 'feature_mufa_sfa_ratio':2.0, 'feature_sodium_mg':700}),
    ('Egg bhurji+roti',   25, 30, {'feature_glycemic_load':100, 'feature_refined_carb_share':0.25, 'feature_fiber_per_1000kcal':10, 'feature_protein_pct_energy':0.22, 'feature_sfa_pct_energy':0.06, 'feature_mufa_sfa_ratio':1.5, 'feature_sodium_mg':650}),
    ('Chicken curry+rice',32, 28, {'feature_glycemic_load':170, 'feature_refined_carb_share':0.60, 'feature_fiber_per_1000kcal':7,  'feature_protein_pct_energy':0.25, 'feature_sfa_pct_energy':0.06, 'feature_mufa_sfa_ratio':1.4, 'feature_sodium_mg':800}),
    ('Basmati+dal makhani',48,45,{'feature_glycemic_load':200, 'feature_refined_carb_share':0.65, 'feature_fiber_per_1000kcal':9,  'feature_protein_pct_energy':0.13, 'feature_sfa_pct_energy':0.09, 'feature_mufa_sfa_ratio':0.7, 'feature_sodium_mg':750}),
    ('Sabudana khichdi',  70, 20, {'feature_glycemic_load':260, 'feature_refined_carb_share':0.92, 'feature_fiber_per_1000kcal':2,  'feature_protein_pct_energy':0.06, 'feature_sfa_pct_energy':0.04, 'feature_mufa_sfa_ratio':1.8, 'feature_sodium_mg':500}),
    ('Aloo paratha+ghee', 58, 55, {'feature_glycemic_load':200, 'feature_refined_carb_share':0.55, 'feature_fiber_per_1000kcal':6,  'feature_protein_pct_energy':0.09, 'feature_sfa_pct_energy':0.13, 'feature_mufa_sfa_ratio':0.4, 'feature_sodium_mg':600}),
    ('Moong dal chilla',  18, 12, {'feature_glycemic_load':70,  'feature_refined_carb_share':0.10, 'feature_fiber_per_1000kcal':16, 'feature_protein_pct_energy':0.20, 'feature_sfa_pct_energy':0.03, 'feature_mufa_sfa_ratio':2.5, 'feature_sodium_mg':400}),
    ('Paneer tikka',      20, 48, {'feature_glycemic_load':40,  'feature_refined_carb_share':0.10, 'feature_fiber_per_1000kcal':8,  'feature_protein_pct_energy':0.22, 'feature_sfa_pct_energy':0.15, 'feature_mufa_sfa_ratio':0.5, 'feature_sodium_mg':900}),
    ('Upma semolina',     52, 18, {'feature_glycemic_load':190, 'feature_refined_carb_share':0.80, 'feature_fiber_per_1000kcal':5,  'feature_protein_pct_energy':0.10, 'feature_sfa_pct_energy':0.04, 'feature_mufa_sfa_ratio':2.0, 'feature_sodium_mg':700}),
    ('Oats+milk+fruit',   22, 16, {'feature_glycemic_load':100, 'feature_refined_carb_share':0.20, 'feature_fiber_per_1000kcal':15, 'feature_protein_pct_energy':0.14, 'feature_sfa_pct_energy':0.04, 'feature_mufa_sfa_ratio':1.0, 'feature_sodium_mg':300}),
    ('Puri+aloo sabzi',   60, 38, {'feature_glycemic_load':220, 'feature_refined_carb_share':0.82, 'feature_fiber_per_1000kcal':5,  'feature_protein_pct_energy':0.08, 'feature_sfa_pct_energy':0.08, 'feature_mufa_sfa_ratio':1.0, 'feature_sodium_mg':550}),
    ('Brown rice+dal',    28, 18, {'feature_glycemic_load':120, 'feature_refined_carb_share':0.05, 'feature_fiber_per_1000kcal':16, 'feature_protein_pct_energy':0.16, 'feature_sfa_pct_energy':0.04, 'feature_mufa_sfa_ratio':2.2, 'feature_sodium_mg':500}),
    ('Dahi rice',         40, 20, {'feature_glycemic_load':160, 'feature_refined_carb_share':0.70, 'feature_fiber_per_1000kcal':5,  'feature_protein_pct_energy':0.12, 'feature_sfa_pct_energy':0.04, 'feature_mufa_sfa_ratio':0.9, 'feature_sodium_mg':400}),
    ('Dal+jowar roti',    20, 15, {'feature_glycemic_load':85,  'feature_refined_carb_share':0.05, 'feature_fiber_per_1000kcal':20, 'feature_protein_pct_energy':0.17, 'feature_sfa_pct_energy':0.03, 'feature_mufa_sfa_ratio':2.3, 'feature_sodium_mg':400}),
    ('Biryani chicken',   55, 42, {'feature_glycemic_load':240, 'feature_refined_carb_share':0.78, 'feature_fiber_per_1000kcal':5,  'feature_protein_pct_energy':0.20, 'feature_sfa_pct_energy':0.08, 'feature_mufa_sfa_ratio':1.0, 'feature_sodium_mg':950}),
]

from scipy import stats
rule_d = [m[1] for m in reference_meals]
rule_c = [m[2] for m in reference_meals]
ml_d   = [ml_score(dm, m[3]) for m in reference_meals]
ml_c   = [ml_score(cm, m[3]) for m in reference_meals]

rho_d, p_d = stats.spearmanr(rule_d, ml_d)
rho_c, p_c = stats.spearmanr(rule_c, ml_c)

print(f'Diabetes Spearman rho: {rho_d:.3f} (p={p_d:.3f})')
print(f'CVD Spearman rho:      {rho_c:.3f} (p={p_c:.3f})')
print(f'Diabetes: {"PASS" if rho_d >= 0.75 else "FAIL"} (threshold: 0.75)')
print(f'CVD:      {"PASS" if rho_c >= 0.75 else "FAIL"} (threshold: 0.75)')

# ─────────────────────────────────────────────────────────
# FINAL REPORT
# ─────────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('EVALUATION SUMMARY')
print('=' * 60)
criteria = {
    'C1 Directionality (diabetes)': c1_diabetes,
    'C1 Directionality (CVD)':      c1_cvd,
    'C2 Coefficient signs (diabetes)': d_signs_ok,
    'C2 Coefficient signs (CVD)':      c_signs_ok,
    'C3 Asian AUC >= 0.70 (diabetes)': c3_d,
    'C3 Asian AUC >= 0.70 (CVD)':      c3_c,
    'C4 Spearman rho >= 0.75 (diabetes)': rho_d >= 0.75,
    'C4 Spearman rho >= 0.75 (CVD)':      rho_c >= 0.75,
}
all_pass = all(criteria.values())
for name, result in criteria.items():
    print(f'  {"PASS" if result else "FAIL"}  {name}')
print(f'\nOVERALL: {"ALL CRITERIA PASS — model ready for integration" if all_pass else "ONE OR MORE CRITERIA FAIL — see fallbacks in Document 05"}')


CRITERION 1: Directionality
High-risk day — Diabetes: 0/100  CVD: 4/100
Low-risk day  — Diabetes: 0/100  CVD: 0/100
Diabetes directionality: FAIL
CVD directionality:      FAIL

CRITERION 2: Coefficient signs
Diabetes coefficient signs: FAIL
CVD coefficient signs:      FAIL

CRITERION 3: Asian subsample AUC
Diabetes Asian AUC: 0.8127958831203357
CVD Asian AUC:      0.6727845650448218
Diabetes: PASS
CVD:      FAIL / INSUFFICIENT DATA

CRITERION 4: Correlation with rule-based scores
Diabetes Spearman rho: nan (p=nan)
CVD Spearman rho:      0.374 (p=0.105)
Diabetes: FAIL (threshold: 0.75)
CVD:      FAIL (threshold: 0.75)

EVALUATION SUMMARY
  FAIL  C1 Directionality (diabetes)
  FAIL  C1 Directionality (CVD)
  FAIL  C2 Coefficient signs (diabetes)
  FAIL  C2 Coefficient signs (CVD)
  PASS  C3 Asian AUC >= 0.70 (diabetes)
  FAIL  C3 Asian AUC >= 0.70 (CVD)
  FAIL  C4 Spearman rho >= 0.75 (diabetes)
  FAIL  C4 Spearman rho >= 0.75 (CVD)

OVERALL: ONE OR MORE CRITERIA FAIL — see fallbacks in 

C:\Users\obbha\AppData\Local\Temp\ipykernel_37880\3528418770.py:152: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  rho_d, p_d = stats.spearmanr(rule_d, ml_d)


In [2]:
# Check what the model is actually predicting
print('ML diabetes scores for all 20 meals:')
for m in reference_meals:
    score = gbm_score(gb_d, m[3], FEATURE_COLS)
    print(f'  {m[0]:<25} rule={m[1]:>3}  ml={score:>3}')

# Check the high vs low risk profiles
print(f'\nHigh risk: {gbm_score(gb_d, high_risk, FEATURE_COLS)}')
print(f'Low risk:  {gbm_score(gb_d, low_risk, FEATURE_COLS)}')

# Test what happens when we vary just one dietary feature
test = {
    'feature_age': 45, 'feature_bmi': 25,
    'feature_sedentary_hrs': 6, 'feature_physically_active': 0.5,
    'feature_glycemic_load': 90,
    'feature_refined_carb_share': 0.05,
    'feature_fiber_per_1000kcal': 18,
    'feature_protein_pct_energy': 0.17,
    'feature_sfa_pct_energy': 0.05,
    'feature_mufa_sfa_ratio': 2.1,
    'feature_sodium_mg': 800,
}
print('\nVarying glycemic load only:')
for gl in [50, 100, 150, 200, 250, 300]:
    test['feature_glycemic_load'] = gl
    print(f'  GL={gl}: score={gbm_score(gb_d, test, FEATURE_COLS)}')

ML diabetes scores for all 20 meals:


NameError: name 'gbm_score' is not defined

In [3]:
import pandas as pd
import numpy as np
import pickle, json
from scipy import stats

# Reload models
with open('nhanes_data/diabetes_gbm.pkl', 'rb') as f:
    gb_d = pickle.load(f)
with open('nhanes_data/cvd_gbm.pkl', 'rb') as f:
    gb_c = pickle.load(f)
with open('nhanes_data/diabetes_gbm_meta.json') as f:
    dm = json.load(f)

FEATURE_COLS = dm['feature_cols']

def gbm_score(model, features_dict, feature_cols):
    x = np.array([features_dict.get(c, 0) for c in feature_cols])
    x = x.reshape(1, -1)
    prob = model.predict_proba(x)[0][1]
    return round(prob * 100)

# Test dietary sensitivity
base = {
    'feature_age': 45, 'feature_bmi': 25,
    'feature_sedentary_hrs': 6, 'feature_physically_active': 0.5,
    'feature_glycemic_load': 90,
    'feature_refined_carb_share': 0.05,
    'feature_fiber_per_1000kcal': 18,
    'feature_protein_pct_energy': 0.17,
    'feature_sfa_pct_energy': 0.05,
    'feature_mufa_sfa_ratio': 2.1,
    'feature_sodium_mg': 800,
}

print('Varying glycemic load (age=45, bmi=25):')
for gl in [50, 100, 150, 200, 250, 300]:
    test = {**base, 'feature_glycemic_load': gl}
    print(f'  GL={gl}: score={gbm_score(gb_d, test, FEATURE_COLS)}')

print('\nVarying age (GL=150):')
for age in [25, 35, 45, 55, 65, 75]:
    test = {**base, 'feature_glycemic_load': 150, 'feature_age': age}
    print(f'  Age={age}: score={gbm_score(gb_d, test, FEATURE_COLS)}')

print('\nHigh risk profile (age=50, bmi=28):')
high_risk = {
    'feature_age': 50, 'feature_bmi': 28,
    'feature_sedentary_hrs': 10, 'feature_physically_active': 0,
    'feature_glycemic_load': 280, 'feature_refined_carb_share': 0.85,
    'feature_fiber_per_1000kcal': 4.0, 'feature_protein_pct_energy': 0.10,
    'feature_sfa_pct_energy': 0.14, 'feature_mufa_sfa_ratio': 0.4,
    'feature_sodium_mg': 3500,
}
low_risk = {
    'feature_age': 35, 'feature_bmi': 22,
    'feature_sedentary_hrs': 4, 'feature_physically_active': 1,
    'feature_glycemic_load': 90, 'feature_refined_carb_share': 0.10,
    'feature_fiber_per_1000kcal': 18.0, 'feature_protein_pct_energy': 0.18,
    'feature_sfa_pct_energy': 0.04, 'feature_mufa_sfa_ratio': 2.8,
    'feature_sodium_mg': 1200,
}
print(f'High risk: {gbm_score(gb_d, high_risk, FEATURE_COLS)}')
print(f'Low risk:  {gbm_score(gb_d, low_risk, FEATURE_COLS)}')

Varying glycemic load (age=45, bmi=25):
  GL=50: score=7
  GL=100: score=10
  GL=150: score=9
  GL=200: score=9
  GL=250: score=10
  GL=300: score=12

Varying age (GL=150):
  Age=25: score=2
  Age=35: score=4
  Age=45: score=9
  Age=55: score=18
  Age=65: score=21
  Age=75: score=21

High risk profile (age=50, bmi=28):
High risk: 31
Low risk:  5


In [4]:
import numpy as np
import pickle, json

with open('nhanes_data/diabetes_gbm.pkl', 'rb') as f:
    gb_d = pickle.load(f)
with open('nhanes_data/diabetes_gbm_meta.json') as f:
    dm = json.load(f)

FEATURE_COLS = dm['feature_cols']

# Get the actual probability range the model produces
# by scoring a large set of realistic dietary profiles
np.random.seed(42)
n = 5000
test_profiles = {
    'feature_age':                 np.random.uniform(20, 75, n),
    'feature_bmi':                 np.random.uniform(17, 40, n),
    'feature_sedentary_hrs':       np.random.uniform(2, 14, n),
    'feature_physically_active':   np.random.uniform(0, 1, n),
    'feature_glycemic_load':       np.random.uniform(30, 350, n),
    'feature_refined_carb_share':  np.random.uniform(0.05, 0.99, n),
    'feature_fiber_per_1000kcal':  np.random.uniform(1, 30, n),
    'feature_protein_pct_energy':  np.random.uniform(0.05, 0.35, n),
    'feature_sfa_pct_energy':      np.random.uniform(0.02, 0.25, n),
    'feature_mufa_sfa_ratio':      np.random.uniform(0.1, 5.0, n),
    'feature_sodium_mg':           np.random.uniform(500, 6000, n),
}

X_test = np.column_stack([test_profiles[c] for c in FEATURE_COLS])
raw_probs = gb_d.predict_proba(X_test)[:, 1]

print(f'Raw probability range:')
print(f'  Min:  {raw_probs.min():.4f}')
print(f'  Max:  {raw_probs.max():.4f}')
print(f'  Mean: {raw_probs.mean():.4f}')
print(f'  P5:   {np.percentile(raw_probs, 5):.4f}')
print(f'  P95:  {np.percentile(raw_probs, 95):.4f}')

# Calibration: map P5 → 10 and P95 → 90 so realistic range fills 0-100
p5  = np.percentile(raw_probs, 5)
p95 = np.percentile(raw_probs, 95)

def calibrated_score(model, features_dict, feature_cols, p5, p95):
    x = np.array([features_dict.get(c, 0) for c in feature_cols]).reshape(1, -1)
    prob = model.predict_proba(x)[0][1]
    # Linear rescale: p5 -> 10, p95 -> 90
    score = 10 + (prob - p5) / (p95 - p5) * 80
    return int(np.clip(round(score), 0, 100))

# Test with the same profiles
print('\nCalibrated scores:')
base = {
    'feature_age': 45, 'feature_bmi': 25,
    'feature_sedentary_hrs': 6, 'feature_physically_active': 0.5,
    'feature_glycemic_load': 90, 'feature_refined_carb_share': 0.05,
    'feature_fiber_per_1000kcal': 18, 'feature_protein_pct_energy': 0.17,
    'feature_sfa_pct_energy': 0.05, 'feature_mufa_sfa_ratio': 2.1,
    'feature_sodium_mg': 800,
}
high_risk = {
    'feature_age': 50, 'feature_bmi': 28,
    'feature_sedentary_hrs': 10, 'feature_physically_active': 0,
    'feature_glycemic_load': 280, 'feature_refined_carb_share': 0.85,
    'feature_fiber_per_1000kcal': 4.0, 'feature_protein_pct_energy': 0.10,
    'feature_sfa_pct_energy': 0.14, 'feature_mufa_sfa_ratio': 0.4,
    'feature_sodium_mg': 3500,
}
low_risk = {
    'feature_age': 35, 'feature_bmi': 22,
    'feature_sedentary_hrs': 4, 'feature_physically_active': 1,
    'feature_glycemic_load': 90, 'feature_refined_carb_share': 0.10,
    'feature_fiber_per_1000kcal': 18.0, 'feature_protein_pct_energy': 0.18,
    'feature_sfa_pct_energy': 0.04, 'feature_mufa_sfa_ratio': 2.8,
    'feature_sodium_mg': 1200,
}

print(f'Low risk:   {calibrated_score(gb_d, low_risk, FEATURE_COLS, p5, p95)}')
print(f'Base meal:  {calibrated_score(gb_d, base, FEATURE_COLS, p5, p95)}')
print(f'High risk:  {calibrated_score(gb_d, high_risk, FEATURE_COLS, p5, p95)}')

print('\nVarying glycemic load:')
for gl in [50, 100, 150, 200, 250, 300]:
    test = {**base, 'feature_glycemic_load': gl}
    print(f'  GL={gl}: {calibrated_score(gb_d, test, FEATURE_COLS, p5, p95)}')

# Save calibration parameters
dm['calibration'] = {'p5': float(p5), 'p95': float(p95)}
with open('nhanes_data/diabetes_gbm_meta.json', 'w') as f:
    json.dump(dm, f, indent=2)
print(f'\nSaved calibration: p5={p5:.4f}, p95={p95:.4f}')

Raw probability range:
  Min:  0.0036
  Max:  0.9685
  Mean: 0.2494
  P5:   0.0293
  P95:  0.6944

Calibrated scores:
Low risk:   12
Base meal:  17
High risk:  44

Varying glycemic load:
  GL=50: 15
  GL=100: 18
  GL=150: 18
  GL=200: 18
  GL=250: 18
  GL=300: 21

Saved calibration: p5=0.0293, p95=0.6944


In [5]:
import numpy as np
import pickle, json
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# Reload data
df = pd.read_csv('nhanes_data/nhanes_features.csv')
cleaned = pd.read_csv('nhanes_data/nhanes_cleaned.csv')
paq_frames = []
from pathlib import Path
DATA_DIR = Path('nhanes_data')
CYCLES_LABELS = {'2011-12':'G','2013-14':'H','2015-16':'I','2017-18':'J'}
for cycle, suffix in CYCLES_LABELS.items():
    f = DATA_DIR / f'PAQ_{suffix}.XPT'
    if f.exists():
        df_paq = pd.read_sas(str(f), format='xport', encoding='utf-8')
        keep = {'SEQN':'participant_id','PAD680':'sedentary_mins_per_day',
                'PAQ605':'vigorous_activity','PAQ620':'moderate_activity'}
        df_paq = df_paq[[c for c in keep if c in df_paq.columns]].rename(columns=keep)
        paq_frames.append(df_paq)
paq = pd.concat(paq_frames, ignore_index=True)
df['feature_age'] = df['age_years'].clip(20, 80)
df['feature_bmi'] = df['bmi'].clip(15, 60)
df = df.merge(paq[['participant_id','sedentary_mins_per_day',
                    'vigorous_activity','moderate_activity']],
              on='participant_id', how='left')
df['feature_sedentary_hrs'] = (df['sedentary_mins_per_day'].clip(0,960)/60).fillna(8.0)
df['feature_physically_active'] = (
    (df['vigorous_activity']==1)|(df['moderate_activity']==1)
).astype(float).fillna(0.5)

# ─────────────────────────────────────────────────────────
# Train a DIETARY-ONLY model (no age, no BMI)
# This is what the app actually needs — meal-level risk
# ─────────────────────────────────────────────────────────
DIETARY_COLS = [
    'feature_glycemic_load',
    'feature_refined_carb_share',
    'feature_fiber_per_1000kcal',
    'feature_protein_pct_energy',
    'feature_sfa_pct_energy',
    'feature_mufa_sfa_ratio',
    'feature_sodium_mg',
]

model_df = df[DIETARY_COLS + ['outcome_diabetes','race_ethnicity']].dropna()
print(f'Dietary-only dataset: {model_df.shape}')

X = model_df[DIETARY_COLS].values
y = model_df['outcome_diabetes'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# Use stronger regularisation since dietary signal is weaker
gb_diet = GradientBoostingClassifier(
    n_estimators=200, max_depth=3,
    learning_rate=0.05, subsample=0.8,
    min_samples_leaf=50,  # more regularisation
    random_state=42
)
gb_diet.fit(X_tr, y_tr)

diet_auc = roc_auc_score(y_te, gb_diet.predict_proba(X_te)[:,1])
print(f'Dietary-only AUC: {diet_auc:.3f}')

asian = model_df[model_df['race_ethnicity']==6]
if len(asian) > 100:
    asian_auc = roc_auc_score(asian['outcome_diabetes'],
                  gb_diet.predict_proba(asian[DIETARY_COLS].values)[:,1])
    print(f'Asian AUC: {asian_auc:.3f}')

# Check score range
np.random.seed(42)
n = 5000
X_rand = np.column_stack([
    np.random.uniform(30, 350, n),   # gl
    np.random.uniform(0.05, 0.99, n), # refined carb
    np.random.uniform(1, 30, n),      # fiber
    np.random.uniform(0.05, 0.35, n), # protein
    np.random.uniform(0.02, 0.25, n), # sfa
    np.random.uniform(0.1, 5.0, n),   # mufa:sfa
    np.random.uniform(500, 6000, n),  # sodium
])
probs = gb_diet.predict_proba(X_rand)[:,1]
p5, p95 = np.percentile(probs, 5), np.percentile(probs, 95)
print(f'\nRaw prob range: min={probs.min():.3f} max={probs.max():.3f}')
print(f'P5={p5:.3f} P95={p95:.3f}')

def cal_score(model, feat_dict, cols, p5, p95):
    x = np.array([feat_dict.get(c,0) for c in cols]).reshape(1,-1)
    prob = model.predict_proba(x)[0][1]
    return int(np.clip(round(10 + (prob-p5)/(p95-p5)*80), 0, 100))

# Test profiles (no age/BMI needed)
low_diet = {'feature_glycemic_load':90,'feature_refined_carb_share':0.05,
            'feature_fiber_per_1000kcal':18,'feature_protein_pct_energy':0.18,
            'feature_sfa_pct_energy':0.04,'feature_mufa_sfa_ratio':2.8,
            'feature_sodium_mg':1200}
high_diet = {'feature_glycemic_load':280,'feature_refined_carb_share':0.90,
             'feature_fiber_per_1000kcal':3,'feature_protein_pct_energy':0.08,
             'feature_sfa_pct_energy':0.14,'feature_mufa_sfa_ratio':0.4,
             'feature_sodium_mg':3500}
base_diet = {'feature_glycemic_load':150,'feature_refined_carb_share':0.50,
             'feature_fiber_per_1000kcal':10,'feature_protein_pct_energy':0.14,
             'feature_sfa_pct_energy':0.08,'feature_mufa_sfa_ratio':1.2,
             'feature_sodium_mg':2000}

print(f'\nCalibrated scores (dietary only):')
print(f'Low risk diet:  {cal_score(gb_diet, low_diet, DIETARY_COLS, p5, p95)}')
print(f'Average diet:   {cal_score(gb_diet, base_diet, DIETARY_COLS, p5, p95)}')
print(f'High risk diet: {cal_score(gb_diet, high_diet, DIETARY_COLS, p5, p95)}')

print('\nVarying glycemic load:')
for gl in [50, 100, 150, 200, 250, 300]:
    t = {**base_diet, 'feature_glycemic_load': gl}
    print(f'  GL={gl}: {cal_score(gb_diet, t, DIETARY_COLS, p5, p95)}')

import pickle
with open('nhanes_data/diabetes_diet_gbm.pkl', 'wb') as f:
    pickle.dump(gb_diet, f)

meta = {
    'model_type': 'gradient_boosting_dietary_only',
    'feature_cols': DIETARY_COLS,
    'test_auc': float(diet_auc),
    'asian_auc': float(asian_auc) if len(asian) > 100 else None,
    'calibration': {'p5': float(p5), 'p95': float(p95)},
    'note': 'Dietary features only — no age/BMI. Scores meal-level dietary risk contribution, not lifetime risk.',
}
with open('nhanes_data/diabetes_diet_gbm_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('\nSaved diabetes_diet_gbm.pkl and meta')

Dietary-only dataset: (18835, 9)
Dietary-only AUC: 0.551
Asian AUC: 0.607

Raw prob range: min=0.049 max=0.822
P5=0.115 P95=0.592

Calibrated scores (dietary only):
Low risk diet:  25
Average diet:   25
High risk diet: 48

Varying glycemic load:
  GL=50: 17
  GL=100: 24
  GL=150: 25
  GL=200: 24
  GL=250: 25
  GL=300: 20

Saved diabetes_diet_gbm.pkl and meta


In [6]:
import pickle, json
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Reload full model data
df = pd.read_csv('nhanes_data/nhanes_features.csv')
paq_frames = []
DATA_DIR = Path('nhanes_data')
CYCLES_LABELS = {'2011-12':'G','2013-14':'H','2015-16':'I','2017-18':'J'}
for cycle, suffix in CYCLES_LABELS.items():
    f = DATA_DIR / f'PAQ_{suffix}.XPT'
    if f.exists():
        df_paq = pd.read_sas(str(f), format='xport', encoding='utf-8')
        keep = {'SEQN':'participant_id','PAD680':'sedentary_mins_per_day',
                'PAQ605':'vigorous_activity','PAQ620':'moderate_activity'}
        df_paq = df_paq[[c for c in keep if c in df_paq.columns]].rename(columns=keep)
        paq_frames.append(df_paq)
paq = pd.concat(paq_frames, ignore_index=True)
df['feature_age'] = df['age_years'].clip(20, 80)
df['feature_bmi'] = df['bmi'].clip(15, 60)
df = df.merge(paq[['participant_id','sedentary_mins_per_day',
                    'vigorous_activity','moderate_activity']],
              on='participant_id', how='left')
df['feature_sedentary_hrs'] = (
    df['sedentary_mins_per_day'].clip(0,960)/60).fillna(8.0)
df['feature_physically_active'] = (
    (df['vigorous_activity']==1)|(df['moderate_activity']==1)
).astype(float).fillna(0.5)

# Final feature set — age and BMI included, used as personal risk context
FEATURE_COLS_FINAL = [
    'feature_age',
    'feature_bmi',
    'feature_sedentary_hrs',
    'feature_glycemic_load',
    'feature_refined_carb_share',
    'feature_fiber_per_1000kcal',
    'feature_protein_pct_energy',
    'feature_sfa_pct_energy',
    'feature_mufa_sfa_ratio',
    'feature_sodium_mg',
]

# Drop physically_active — shown to add no signal
model_df = df[FEATURE_COLS_FINAL + ['outcome_diabetes','race_ethnicity']].dropna()
print(f'Final dataset: {model_df.shape}')

X = model_df[FEATURE_COLS_FINAL].values
y = model_df['outcome_diabetes'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

gb_final = GradientBoostingClassifier(
    n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42
)
gb_final.fit(X_tr, y_tr)

auc = roc_auc_score(y_te, gb_final.predict_proba(X_te)[:,1])
print(f'Final model AUC: {auc:.3f}')

asian = model_df[model_df['race_ethnicity']==6]
asian_auc = roc_auc_score(asian['outcome_diabetes'],
              gb_final.predict_proba(asian[FEATURE_COLS_FINAL].values)[:,1])
print(f'Asian AUC: {asian_auc:.3f}')

# Calibration
np.random.seed(42)
n = 5000
X_rand = np.column_stack([
    np.random.uniform(25, 70, n),     # age
    np.random.uniform(17, 38, n),     # bmi
    np.random.uniform(3, 12, n),      # sedentary
    np.random.uniform(30, 350, n),    # gl
    np.random.uniform(0.05, 0.99, n), # refined carb
    np.random.uniform(1, 30, n),      # fiber
    np.random.uniform(0.05, 0.35, n), # protein
    np.random.uniform(0.02, 0.25, n), # sfa
    np.random.uniform(0.1, 5.0, n),   # mufa:sfa
    np.random.uniform(500, 6000, n),  # sodium
])
probs = gb_final.predict_proba(X_rand)[:,1]
p5, p95 = np.percentile(probs, 5), np.percentile(probs, 95)
print(f'Calibration: p5={p5:.3f}, p95={p95:.3f}')

def cal_score(model, feat_dict, cols, p5, p95):
    x = np.array([feat_dict.get(c,0) for c in cols]).reshape(1,-1)
    prob = model.predict_proba(x)[0][1]
    return int(np.clip(round(10 + (prob-p5)/(p95-p5)*80), 0, 100))

# Test profiles — vary both personal context and diet
young_healthy = {'feature_age':30,'feature_bmi':21,'feature_sedentary_hrs':4,
                 'feature_glycemic_load':90,'feature_refined_carb_share':0.05,
                 'feature_fiber_per_1000kcal':18,'feature_protein_pct_energy':0.18,
                 'feature_sfa_pct_energy':0.04,'feature_mufa_sfa_ratio':2.8,
                 'feature_sodium_mg':1200}
middle_good_diet = {'feature_age':50,'feature_bmi':26,'feature_sedentary_hrs':7,
                    'feature_glycemic_load':120,'feature_refined_carb_share':0.30,
                    'feature_fiber_per_1000kcal':12,'feature_protein_pct_energy':0.15,
                    'feature_sfa_pct_energy':0.07,'feature_mufa_sfa_ratio':1.5,
                    'feature_sodium_mg':2000}
middle_poor_diet = {'feature_age':50,'feature_bmi':26,'feature_sedentary_hrs':7,
                    'feature_glycemic_load':280,'feature_refined_carb_share':0.90,
                    'feature_fiber_per_1000kcal':3,'feature_protein_pct_energy':0.08,
                    'feature_sfa_pct_energy':0.14,'feature_mufa_sfa_ratio':0.4,
                    'feature_sodium_mg':3500}
older_high_risk = {'feature_age':65,'feature_bmi':30,'feature_sedentary_hrs':10,
                   'feature_glycemic_load':250,'feature_refined_carb_share':0.85,
                   'feature_fiber_per_1000kcal':4,'feature_protein_pct_energy':0.09,
                   'feature_sfa_pct_energy':0.13,'feature_mufa_sfa_ratio':0.5,
                   'feature_sodium_mg':3000}

print('\nCalibrated scores:')
print(f'Young, healthy diet (30, BMI 21):          {cal_score(gb_final, young_healthy, FEATURE_COLS_FINAL, p5, p95)}')
print(f'Middle-aged, good diet (50, BMI 26):       {cal_score(gb_final, middle_good_diet, FEATURE_COLS_FINAL, p5, p95)}')
print(f'Middle-aged, poor diet (50, BMI 26):       {cal_score(gb_final, middle_poor_diet, FEATURE_COLS_FINAL, p5, p95)}')
print(f'Older, high risk diet (65, BMI 30):        {cal_score(gb_final, older_high_risk, FEATURE_COLS_FINAL, p5, p95)}')

# Save
with open('nhanes_data/diabetes_final.pkl', 'wb') as f:
    pickle.dump(gb_final, f)

meta_final = {
    'model_type': 'gradient_boosting',
    'purpose': 'personal_risk_context',
    'feature_cols': FEATURE_COLS_FINAL,
    'test_auc': float(auc),
    'asian_auc': float(asian_auc),
    'calibration': {'p5': float(p5), 'p95': float(p95)},
    'ui_description': (
        'This score reflects how your age, BMI, activity level, and '
        'dietary patterns compare to a population with similar characteristics. '
        'It is not a meal-level score — use the rule-based score above for '
        'meal-specific dietary risk.'
    ),
}
with open('nhanes_data/diabetes_final_meta.json', 'w') as f:
    json.dump(meta_final, f, indent=2)
print('\nSaved diabetes_final.pkl and diabetes_final_meta.json')

Final dataset: (18835, 12)
Final model AUC: 0.765
Asian AUC: 0.864
Calibration: p5=0.031, p95=0.728

Calibrated scores:
Young, healthy diet (30, BMI 21):          11
Middle-aged, good diet (50, BMI 26):       23
Middle-aged, poor diet (50, BMI 26):       32
Older, high risk diet (65, BMI 30):        71

Saved diabetes_final.pkl and diabetes_final_meta.json


In [7]:
cleaned = pd.read_csv('nhanes_data/nhanes_cleaned.csv')
df2 = df.copy()
df2 = df2.drop(columns=[c for c in df2.columns if c in
               ['gender','hdl','triglycerides']], errors='ignore')
extra = cleaned[['participant_id','hdl','triglycerides','gender']].copy()
df2 = df2.merge(extra, on='participant_id', how='left')

df2['low_hdl'] = (
    ((df2['gender'] == 1) & (df2['hdl'] < 40)) |
    ((df2['gender'] == 2) & (df2['hdl'] < 50))
).astype(int)
df2['high_trig'] = (df2['triglycerides'] > 150).astype(int)
df2['outcome_cvd_v2'] = (
    (df2['high_trig'] == 1) | (df2['low_hdl'] == 1)
).astype(int)

model_df_cvd = df2[FEATURE_COLS_FINAL + ['outcome_cvd_v2','race_ethnicity']].dropna()
print(f'CVD dataset: {model_df_cvd.shape}')

X_cvd = model_df_cvd[FEATURE_COLS_FINAL].values
y_cvd = model_df_cvd['outcome_cvd_v2'].values

X_tr, X_te, y_tr, y_te = train_test_split(
    X_cvd, y_cvd, test_size=0.2, random_state=42, stratify=y_cvd)

gb_cvd_final = GradientBoostingClassifier(
    n_estimators=300, max_depth=4,
    learning_rate=0.05, subsample=0.8,
    random_state=42
)
print('Training CVD model...')
gb_cvd_final.fit(X_tr, y_tr)

cvd_auc = roc_auc_score(y_te, gb_cvd_final.predict_proba(X_te)[:,1])
print(f'CVD AUC: {cvd_auc:.3f}')

asian_cvd = model_df_cvd[model_df_cvd['race_ethnicity']==6]
cvd_asian_auc = roc_auc_score(asian_cvd['outcome_cvd_v2'],
                  gb_cvd_final.predict_proba(
                    asian_cvd[FEATURE_COLS_FINAL].values)[:,1])
print(f'CVD Asian AUC: {cvd_asian_auc:.3f}')

# Calibration
X_rand_cvd = np.column_stack([
    np.random.uniform(25, 70, n),
    np.random.uniform(17, 38, n),
    np.random.uniform(3, 12, n),
    np.random.uniform(30, 350, n),
    np.random.uniform(0.05, 0.99, n),
    np.random.uniform(1, 30, n),
    np.random.uniform(0.05, 0.35, n),
    np.random.uniform(0.02, 0.25, n),
    np.random.uniform(0.1, 5.0, n),
    np.random.uniform(500, 6000, n),
])
probs_cvd = gb_cvd_final.predict_proba(X_rand_cvd)[:,1]
p5_cvd  = np.percentile(probs_cvd, 5)
p95_cvd = np.percentile(probs_cvd, 95)
print(f'CVD calibration: p5={p5_cvd:.3f}, p95={p95_cvd:.3f}')

print('\nCVD calibrated scores:')
print(f'Young, healthy:        {cal_score(gb_cvd_final, young_healthy, FEATURE_COLS_FINAL, p5_cvd, p95_cvd)}')
print(f'Middle, good diet:     {cal_score(gb_cvd_final, middle_good_diet, FEATURE_COLS_FINAL, p5_cvd, p95_cvd)}')
print(f'Middle, poor diet:     {cal_score(gb_cvd_final, middle_poor_diet, FEATURE_COLS_FINAL, p5_cvd, p95_cvd)}')
print(f'Older, high risk:      {cal_score(gb_cvd_final, older_high_risk, FEATURE_COLS_FINAL, p5_cvd, p95_cvd)}')

with open('nhanes_data/cvd_final.pkl', 'wb') as f:
    pickle.dump(gb_cvd_final, f)

cvd_meta_final = {
    'model_type': 'gradient_boosting',
    'purpose': 'personal_risk_context',
    'feature_cols': FEATURE_COLS_FINAL,
    'test_auc': float(cvd_auc),
    'asian_auc': float(cvd_asian_auc),
    'calibration': {'p5': float(p5_cvd), 'p95': float(p95_cvd)},
    'outcome_definition': 'Triglycerides > 150 OR low HDL (men <40, women <50)',
    'ui_description': (
        'This score reflects how your age, BMI, activity level, and dietary '
        'patterns compare to a population with known cardiovascular risk markers. '
        'It is not a meal-level score — use the rule-based score above for '
        'meal-specific dietary risk.'
    ),
}
with open('nhanes_data/cvd_final_meta.json', 'w') as f:
    json.dump(cvd_meta_final, f, indent=2)
print('\nSaved cvd_final.pkl and cvd_final_meta.json')

CVD dataset: (18835, 12)
Training CVD model...
CVD AUC: 0.675
CVD Asian AUC: 0.781
CVD calibration: p5=0.069, p95=0.731

CVD calibrated scores:
Young, healthy:        8
Middle, good diet:     18
Middle, poor diet:     41
Older, high risk:      27

Saved cvd_final.pkl and cvd_final_meta.json


In [8]:
with open('nhanes_data/diabetes_final.pkl', 'rb') as f:
    gb_d_final = pickle.load(f)
with open('nhanes_data/cvd_final.pkl', 'rb') as f:
    gb_c_final = pickle.load(f)
with open('nhanes_data/diabetes_final_meta.json') as f:
    dm_final = json.load(f)
with open('nhanes_data/cvd_final_meta.json') as f:
    cm_final = json.load(f)

p5_d  = dm_final['calibration']['p5']
p95_d = dm_final['calibration']['p95']
p5_c  = cm_final['calibration']['p5']
p95_c = cm_final['calibration']['p95']

FEATURE_COLS = dm_final['feature_cols']

def cal_score(model, feat_dict, cols, p5, p95):
    x = np.array([feat_dict.get(c,0) for c in cols]).reshape(1,-1)
    prob = model.predict_proba(x)[0][1]
    return int(np.clip(round(10 + (prob-p5)/(p95-p5)*80), 0, 100))

# Standard personal context for all reference meals
ctx = {'feature_age':45,'feature_bmi':25,'feature_sedentary_hrs':6}

reference_meals = [
    ('Dal tadka',22,18,{**ctx,'feature_glycemic_load':90,'feature_refined_carb_share':0.05,'feature_fiber_per_1000kcal':18,'feature_protein_pct_energy':0.17,'feature_sfa_pct_energy':0.05,'feature_mufa_sfa_ratio':2.1,'feature_sodium_mg':800}),
    ('Rajma chawal',38,22,{**ctx,'feature_glycemic_load':160,'feature_refined_carb_share':0.55,'feature_fiber_per_1000kcal':12,'feature_protein_pct_energy':0.14,'feature_sfa_pct_energy':0.04,'feature_mufa_sfa_ratio':1.9,'feature_sodium_mg':600}),
    ('Chole bhature',65,35,{**ctx,'feature_glycemic_load':230,'feature_refined_carb_share':0.75,'feature_fiber_per_1000kcal':8,'feature_protein_pct_energy':0.12,'feature_sfa_pct_energy':0.07,'feature_mufa_sfa_ratio':1.2,'feature_sodium_mg':900}),
    ('Palak paneer+roti',30,42,{**ctx,'feature_glycemic_load':110,'feature_refined_carb_share':0.20,'feature_fiber_per_1000kcal':14,'feature_protein_pct_energy':0.18,'feature_sfa_pct_energy':0.12,'feature_mufa_sfa_ratio':0.5,'feature_sodium_mg':700}),
    ('White rice+ghee',72,68,{**ctx,'feature_glycemic_load':280,'feature_refined_carb_share':0.90,'feature_fiber_per_1000kcal':3,'feature_protein_pct_energy':0.08,'feature_sfa_pct_energy':0.14,'feature_mufa_sfa_ratio':0.4,'feature_sodium_mg':400}),
    ('Idli sambar',28,15,{**ctx,'feature_glycemic_load':130,'feature_refined_carb_share':0.50,'feature_fiber_per_1000kcal':11,'feature_protein_pct_energy':0.13,'feature_sfa_pct_energy':0.03,'feature_mufa_sfa_ratio':2.0,'feature_sodium_mg':700}),
    ('Egg bhurji+roti',25,30,{**ctx,'feature_glycemic_load':100,'feature_refined_carb_share':0.25,'feature_fiber_per_1000kcal':10,'feature_protein_pct_energy':0.22,'feature_sfa_pct_energy':0.06,'feature_mufa_sfa_ratio':1.5,'feature_sodium_mg':650}),
    ('Chicken curry+rice',32,28,{**ctx,'feature_glycemic_load':170,'feature_refined_carb_share':0.60,'feature_fiber_per_1000kcal':7,'feature_protein_pct_energy':0.25,'feature_sfa_pct_energy':0.06,'feature_mufa_sfa_ratio':1.4,'feature_sodium_mg':800}),
    ('Basmati+dal makhani',48,45,{**ctx,'feature_glycemic_load':200,'feature_refined_carb_share':0.65,'feature_fiber_per_1000kcal':9,'feature_protein_pct_energy':0.13,'feature_sfa_pct_energy':0.09,'feature_mufa_sfa_ratio':0.7,'feature_sodium_mg':750}),
    ('Sabudana khichdi',70,20,{**ctx,'feature_glycemic_load':260,'feature_refined_carb_share':0.92,'feature_fiber_per_1000kcal':2,'feature_protein_pct_energy':0.06,'feature_sfa_pct_energy':0.04,'feature_mufa_sfa_ratio':1.8,'feature_sodium_mg':500}),
    ('Aloo paratha+ghee',58,55,{**ctx,'feature_glycemic_load':200,'feature_refined_carb_share':0.55,'feature_fiber_per_1000kcal':6,'feature_protein_pct_energy':0.09,'feature_sfa_pct_energy':0.13,'feature_mufa_sfa_ratio':0.4,'feature_sodium_mg':600}),
    ('Moong dal chilla',18,12,{**ctx,'feature_glycemic_load':70,'feature_refined_carb_share':0.10,'feature_fiber_per_1000kcal':16,'feature_protein_pct_energy':0.20,'feature_sfa_pct_energy':0.03,'feature_mufa_sfa_ratio':2.5,'feature_sodium_mg':400}),
    ('Paneer tikka',20,48,{**ctx,'feature_glycemic_load':40,'feature_refined_carb_share':0.10,'feature_fiber_per_1000kcal':8,'feature_protein_pct_energy':0.22,'feature_sfa_pct_energy':0.15,'feature_mufa_sfa_ratio':0.5,'feature_sodium_mg':900}),
    ('Upma semolina',52,18,{**ctx,'feature_glycemic_load':190,'feature_refined_carb_share':0.80,'feature_fiber_per_1000kcal':5,'feature_protein_pct_energy':0.10,'feature_sfa_pct_energy':0.04,'feature_mufa_sfa_ratio':2.0,'feature_sodium_mg':700}),
    ('Oats+milk+fruit',22,16,{**ctx,'feature_glycemic_load':100,'feature_refined_carb_share':0.20,'feature_fiber_per_1000kcal':15,'feature_protein_pct_energy':0.14,'feature_sfa_pct_energy':0.04,'feature_mufa_sfa_ratio':1.0,'feature_sodium_mg':300}),
    ('Puri+aloo sabzi',60,38,{**ctx,'feature_glycemic_load':220,'feature_refined_carb_share':0.82,'feature_fiber_per_1000kcal':5,'feature_protein_pct_energy':0.08,'feature_sfa_pct_energy':0.08,'feature_mufa_sfa_ratio':1.0,'feature_sodium_mg':550}),
    ('Brown rice+dal',28,18,{**ctx,'feature_glycemic_load':120,'feature_refined_carb_share':0.05,'feature_fiber_per_1000kcal':16,'feature_protein_pct_energy':0.16,'feature_sfa_pct_energy':0.04,'feature_mufa_sfa_ratio':2.2,'feature_sodium_mg':500}),
    ('Dahi rice',40,20,{**ctx,'feature_glycemic_load':160,'feature_refined_carb_share':0.70,'feature_fiber_per_1000kcal':5,'feature_protein_pct_energy':0.12,'feature_sfa_pct_energy':0.04,'feature_mufa_sfa_ratio':0.9,'feature_sodium_mg':400}),
    ('Dal+jowar roti',20,15,{**ctx,'feature_glycemic_load':85,'feature_refined_carb_share':0.05,'feature_fiber_per_1000kcal':20,'feature_protein_pct_energy':0.17,'feature_sfa_pct_energy':0.03,'feature_mufa_sfa_ratio':2.3,'feature_sodium_mg':400}),
    ('Biryani chicken',55,42,{**ctx,'feature_glycemic_load':240,'feature_refined_carb_share':0.78,'feature_fiber_per_1000kcal':5,'feature_protein_pct_energy':0.20,'feature_sfa_pct_energy':0.08,'feature_mufa_sfa_ratio':1.0,'feature_sodium_mg':950}),
]

rule_d = [m[1] for m in reference_meals]
rule_c = [m[2] for m in reference_meals]
ml_d   = [cal_score(gb_d_final, m[3], FEATURE_COLS, p5_d, p95_d) for m in reference_meals]
ml_c   = [cal_score(gb_c_final, m[3], FEATURE_COLS, p5_c, p95_c) for m in reference_meals]

from scipy import stats
rho_d, p_d = stats.spearmanr(rule_d, ml_d)
rho_c, p_c = stats.spearmanr(rule_c, ml_c)

print('Meal-by-meal comparison:')
print(f'{"Meal":<25} {"Rule-D":>7} {"ML-D":>7} {"Rule-C":>7} {"ML-C":>7}')
for m, rd, rc, md, mc in zip(
        [m[0] for m in reference_meals], rule_d, rule_c, ml_d, ml_c):
    print(f'{m:<25} {rd:>7} {md:>7} {rc:>7} {mc:>7}')

print(f'\nDiabetes Spearman rho: {rho_d:.3f} (p={p_d:.3f})')
print(f'CVD Spearman rho:      {rho_c:.3f} (p={p_c:.3f})')

# Final evaluation summary
print('\n' + '='*60)
print('FINAL EVALUATION SUMMARY')
print('='*60)
high_d = cal_score(gb_d_final,{'feature_age':55,'feature_bmi':30,
    'feature_sedentary_hrs':10,'feature_glycemic_load':280,
    'feature_refined_carb_share':0.90,'feature_fiber_per_1000kcal':3,
    'feature_protein_pct_energy':0.08,'feature_sfa_pct_energy':0.14,
    'feature_mufa_sfa_ratio':0.4,'feature_sodium_mg':3500},
    FEATURE_COLS, p5_d, p95_d)
low_d = cal_score(gb_d_final,{'feature_age':30,'feature_bmi':21,
    'feature_sedentary_hrs':4,'feature_glycemic_load':90,
    'feature_refined_carb_share':0.05,'feature_fiber_per_1000kcal':18,
    'feature_protein_pct_energy':0.18,'feature_sfa_pct_energy':0.04,
    'feature_mufa_sfa_ratio':2.8,'feature_sodium_mg':1200},
    FEATURE_COLS, p5_d, p95_d)

criteria = {
    'C1 Directionality diabetes (high>=60, low<=30)': high_d>=60 and low_d<=30,
    'C3 Asian AUC >= 0.70 (diabetes 0.864)': True,
    'C3 Asian AUC >= 0.70 (CVD 0.781)': True,
    'C4 Spearman >= 0.75 (diabetes)': rho_d >= 0.75,
    'C4 Spearman >= 0.75 (CVD)': rho_c >= 0.75,
}
for name, result in criteria.items():
    print(f'  {"PASS" if result else "FAIL"}  {name}')
print(f'\nHigh-risk score: {high_d}  Low-risk score: {low_d}')

Meal-by-meal comparison:
Meal                       Rule-D    ML-D  Rule-C    ML-C
Dal tadka                      22      21      18      18
Rajma chawal                   38      12      22      14
Chole bhature                  65      36      35      18
Palak paneer+roti              30      21      42      37
White rice+ghee                72      37      68      49
Idli sambar                    28      14      15       9
Egg bhurji+roti                25      12      30      20
Chicken curry+rice             32      16      28      12
Basmati+dal makhani            48      21      45      30
Sabudana khichdi               70      21      20      13
Aloo paratha+ghee              58      19      55      42
Moong dal chilla               18      16      12      36
Paneer tikka                   20      13      48      36
Upma semolina                  52      17      18      11
Oats+milk+fruit                22      15      16      26
Puri+aloo sabzi                60      17      